<a href="https://colab.research.google.com/github/davidanbernal/ia/blob/main/actividad3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# APRENDIZAJE SUPERVISADO DESDE BASE DE CONOCIMIENTO
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# ------------------------------------------
# 1. BASE DE CONOCIMIENTO (MISMA QUE ANTES)
# ------------------------------------------
red_transporte = {
    "Portal Norte": [("Calle 100", 10), ("Usaquén", 15)],
    "Calle 100": [("Portal Norte", 10), ("Héroes", 8)],
    "Héroes": [("Calle 100", 8), ("Calle 72", 7)],
    "Calle 72": [("Héroes", 7), ("Calle 45", 9), ("Chapinero", 6)],
    "Calle 45": [("Calle 72", 9), ("Av. Jiménez", 10)],
    "Av. Jiménez": [("Calle 45", 10), ("Centro", 5)],
    "Centro": [("Av. Jiménez", 5), ("Museo Nacional", 7)],
    "Museo Nacional": [("Centro", 7)],
    "Chapinero": [("Calle 72", 6)],
    "Usaquén": [("Portal Norte", 15)]
}

# ------------------------------------------
# 2. CONVERTIR A DATASET
# ------------------------------------------
datos = []

for origen, conexiones in red_transporte.items():
    for destino, tiempo in conexiones:
        datos.append({
            "origen": origen,
            "destino": destino,
            "tiempo_min": tiempo
        })

df = pd.DataFrame(datos)

print("DATASET GENERADO DESDE LA BASE DE CONOCIMIENTO:")
print(df)

# ------------------------------------------
# 3. VARIABLES
# ------------------------------------------
X = df[["origen", "destino"]]
y = df["tiempo_min"]

# ------------------------------------------
# 4. DIVISIÓN DE DATOS
# ------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------------
# 5. PREPROCESAMIENTO
# ------------------------------------------
preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["origen", "destino"])
    ]
)

# ------------------------------------------
# 6. MODELO
# ------------------------------------------
modelo = Pipeline([
    ("prep", preprocesador),
    ("modelo", LinearRegression())
])

# ------------------------------------------
# 7. ENTRENAMIENTO
# ------------------------------------------
modelo.fit(X_train, y_train)

# ------------------------------------------
# 8. EVALUACIÓN
# ------------------------------------------
pred = modelo.predict(X_test)

print("\nRESULTADOS DE PRUEBA:")
for real, estimado in zip(y_test, pred):
    print(f"Real: {real} min | Predicción: {round(estimado,2)} min")

error = mean_absolute_error(y_test, pred)
print("\nError promedio:", round(error, 2), "minutos")

# ------------------------------------------
# 9. PREDICCIÓN NUEVA
# ------------------------------------------
nuevo = pd.DataFrame([
    {"origen": "Portal Norte", "destino": "Centro"}
])

resultado = modelo.predict(nuevo)

print("\nPREDICCIÓN NUEVA:")
print("Origen:", nuevo["origen"][0])
print("Destino:", nuevo["destino"][0])
print("Tiempo estimado:", round(resultado[0], 2), "min")

DATASET GENERADO DESDE LA BASE DE CONOCIMIENTO:
            origen         destino  tiempo_min
0     Portal Norte       Calle 100          10
1     Portal Norte         Usaquén          15
2        Calle 100    Portal Norte          10
3        Calle 100          Héroes           8
4           Héroes       Calle 100           8
5           Héroes        Calle 72           7
6         Calle 72          Héroes           7
7         Calle 72        Calle 45           9
8         Calle 72       Chapinero           6
9         Calle 45        Calle 72           9
10        Calle 45     Av. Jiménez          10
11     Av. Jiménez        Calle 45          10
12     Av. Jiménez          Centro           5
13          Centro     Av. Jiménez           5
14          Centro  Museo Nacional           7
15  Museo Nacional          Centro           7
16       Chapinero        Calle 72           6
17         Usaquén    Portal Norte          15

RESULTADOS DE PRUEBA:
Real: 10 min | Predicción: 8.32 min
